In [1]:
import pandas as pd
import numpy as np

# Load Stage 08 patient-phase features
df = pd.read_csv(
    r"C:\Users\HP\Desktop\MSIMI_Project\data\features\08_patient_phase_features.csv"
)

# Select 500 Hz records
df_500 = df[df["sampling_rate"] == 500].copy()

print("Dataset Shape:", df_500.shape)
print("Unique Patients:", df_500["subject_id"].nunique())

Dataset Shape: (80, 264)
Unique Patients: 80


In [2]:
metadata_cols = [
    "subject_id",
    "sampling_rate",
    "is_unlabelled_holdout",
    "stress_recording_status",
    "stage06_candidate_nn_status"
]

feature_cols = [
    col for col in df_500.columns
    if col not in metadata_cols
]

print("Total Features:", len(feature_cols))

for prefix in ["rest_", "stress_", "recover_", "t1_", "t2_"]:
    cols = [c for c in feature_cols if c.startswith(prefix)]

    print(f"\n{prefix} Features: {len(cols)}")
    print(cols)

Total Features: 259

rest_ Features: 67
['rest_min_st60_global_adc', 'rest_min_st60_inferior_adc', 'rest_min_st60_anterior_adc', 'rest_min_st60_lateral_adc', 'rest_qrs_axis_degrees', 'rest_raw_detector_rr_mean_ms', 'rest_raw_detector_rr_rmssd_ms', 'rest_I_st60_adc', 'rest_I_st80_adc', 'rest_I_st_slope_adc_per_ms', 'rest_I_t_amp_adc', 'rest_I_qrs_dur_ms', 'rest_II_st60_adc', 'rest_II_st80_adc', 'rest_II_st_slope_adc_per_ms', 'rest_II_t_amp_adc', 'rest_II_qrs_dur_ms', 'rest_III_st60_adc', 'rest_III_st80_adc', 'rest_III_st_slope_adc_per_ms', 'rest_III_t_amp_adc', 'rest_III_qrs_dur_ms', 'rest_aVR_st60_adc', 'rest_aVR_st80_adc', 'rest_aVR_st_slope_adc_per_ms', 'rest_aVR_t_amp_adc', 'rest_aVR_qrs_dur_ms', 'rest_aVL_st60_adc', 'rest_aVL_st80_adc', 'rest_aVL_st_slope_adc_per_ms', 'rest_aVL_t_amp_adc', 'rest_aVL_qrs_dur_ms', 'rest_aVF_st60_adc', 'rest_aVF_st80_adc', 'rest_aVF_st_slope_adc_per_ms', 'rest_aVF_t_amp_adc', 'rest_aVF_qrs_dur_ms', 'rest_V1_st60_adc', 'rest_V1_st80_adc', 'rest_V1_st_s

In [3]:
missing_summary = pd.DataFrame({
    "Missing_Count": df_500[feature_cols].isnull().sum(),
    "Missing_Percentage": (
        df_500[feature_cols].isnull().mean() * 100
    ).round(2)
})

missing_summary = missing_summary[
    missing_summary["Missing_Count"] > 0
].sort_values("Missing_Percentage", ascending=False)

print("Completely Empty Features:")
print(
    missing_summary[
        missing_summary["Missing_Percentage"] == 100
    ]
)

print("\nAll Features Having Missing Values:")
display(missing_summary)

Completely Empty Features:
                                 Missing_Count  Missing_Percentage
rest_raw_detector_rr_mean_ms                80               100.0
stress_raw_detector_rr_mean_ms              80               100.0
recover_raw_detector_rr_mean_ms             80               100.0

All Features Having Missing Values:


,Missing_Count,Missing_Percentage
rest_raw_detector_rr_mean_ms,80,100.00
stress_raw_detector_rr_mean_ms,80,100.00
recover_raw_detector_rr_mean_ms,80,100.00
t2_hrri_pct,67,83.75


In [4]:
# Keep development records only
df_dev = df_500[
    df_500["is_unlabelled_holdout"] == False
].copy()

# Remove completely empty features and HRRI temporarily
exclude_features = [
    "rest_raw_detector_rr_mean_ms",
    "stress_raw_detector_rr_mean_ms",
    "recover_raw_detector_rr_mean_ms",
    "t2_hrri_pct"
]

candidate_features = [
    col for col in feature_cols
    if col not in exclude_features
]

X_check = df_dev[candidate_features]

# Check constant features
constant_features = [
    col for col in X_check.columns
    if X_check[col].nunique(dropna=False) <= 1
]

# Check duplicate feature columns
duplicate_features = X_check.columns[
    X_check.T.duplicated()
].tolist()

print("Development Records:", len(df_dev))
print("Candidate Features:", len(candidate_features))

print("\nConstant Features:", len(constant_features))
print(constant_features)

print("\nDuplicate Features:", len(duplicate_features))
print(duplicate_features)

Development Records: 78
Candidate Features: 255

Constant Features: 0
[]

Duplicate Features: 0
[]


In [5]:
import os

project_path = r"C:\Users\HP\MSIMI Detection"

for root, dirs, files in os.walk(project_path):
    dirs[:] = [
        d for d in dirs
        if d not in ["__pycache__", ".ipynb_checkpoints"]
    ]

    for file in files:
        if file.endswith((".csv", ".xlsx", ".json")):
            print(os.path.join(root, file))

C:\Users\HP\MSIMI Detection\results\all80_first30s_lead0_QC.csv
C:\Users\HP\MSIMI Detection\results\ANOCA_80_ECG_file_availability.csv
C:\Users\HP\MSIMI Detection\results\phase_wise_QC_summary.csv
C:\Users\HP\MSIMI Detection\results\pilot_5subjects_30s.csv
C:\Users\HP\MSIMI Detection\results\pilot_5subjects_30s_QC.csv
C:\Users\HP\MSIMI Detection\results\pilot_5subjects_30s_QC_summary.csv
C:\Users\HP\MSIMI Detection\results\pilot_5subjects_30s_v2.csv
C:\Users\HP\MSIMI Detection\results\pilot_5subjects_personalized_response.csv
C:\Users\HP\MSIMI Detection\results\pilot_5subjects_Rpeak_comparison.csv
C:\Users\HP\MSIMI Detection\results\pilot_5subjects_RR_screening.csv
C:\Users\HP\MSIMI Detection\results\pilot_5subjects_RR_summary.csv
C:\Users\HP\MSIMI Detection\results\pilot_5subjects_signal_QC.csv
C:\Users\HP\MSIMI Detection\results\pilot_Rpeak_detector_comparison.csv
C:\Users\HP\MSIMI Detection\results\pilot_subjectwise_HR_changes.csv
C:\Users\HP\MSIMI Detection\results\point10_all80_de

In [6]:
import os

project_path = r"C:\Users\HP\MSIMI Detection"

keywords = [
    "label",
    "clinical",
    "diagnosis",
    "msimi",
    "annotation",
    "ground_truth",
    "outcome",
    "ischemia",
    "patient"
]

matches = []

for root, dirs, files in os.walk(project_path):
    dirs[:] = [
        d for d in dirs
        if d not in ["__pycache__", ".ipynb_checkpoints"]
    ]

    for file in files:
        if file.lower().endswith((".csv", ".xlsx", ".json")):
            if any(word in file.lower() for word in keywords):
                matches.append(os.path.join(root, file))

print("Possible Label-Related Files:\n")

for file in matches:
    print(file)

print("\nTotal matching files:", len(matches))

Possible Label-Related Files:

C:\Users\HP\MSIMI Detection\results\point10_annotation_file_search.csv
C:\Users\HP\MSIMI Detection\results\point11_independent_reference_annotation_template.csv
C:\Users\HP\MSIMI Detection\results\point11_reference_validation_and_clinical_review_sheet.csv

Total matching files: 3


In [7]:
import pandas as pd
import os

files_to_check = [
    r"C:\Users\HP\MSIMI Detection\results\point10_annotation_file_search.csv",
    r"C:\Users\HP\MSIMI Detection\results\point11_independent_reference_annotation_template.csv",
    r"C:\Users\HP\MSIMI Detection\results\point11_reference_validation_and_clinical_review_sheet.csv"
]

for file in files_to_check:

    temp = pd.read_csv(file)

    print("\n" + "=" * 80)
    print("FILE:", os.path.basename(file))
    print("SHAPE:", temp.shape)

    print("\nCOLUMNS:")
    print(temp.columns.tolist())

    print("\nFIRST 5 ROWS:")
    display(temp.head())


FILE: point10_annotation_file_search.csv
SHAPE: (5, 4)

COLUMNS:
['Folder', 'File', 'Extension', 'Filename_match']

FIRST 5 ROWS:


,Folder,File,Extension,Filename_match
0,C:\Users\HP\MSIMI Detection\results,C:\Users\HP\MSIMI Detection\results\pilot_5sub...,.csv,True
1,C:\Users\HP\MSIMI Detection\results,C:\Users\HP\MSIMI Detection\results\pilot_Rpea...,.csv,True
2,C:\Users\HP\MSIMI Detection\results,C:\Users\HP\MSIMI Detection\results\sub001_res...,.csv,True
3,C:\Users\HP\MSIMI Detection\results,C:\Users\HP\MSIMI Detection\results\subject001...,.csv,True
4,C:\Users\HP\MSIMI Detection\results,C:\Users\HP\MSIMI Detection\results\.ipynb_che...,.csv,True



FILE: point11_independent_reference_annotation_template.csv
SHAPE: (240, 5)

COLUMNS:
['Subject', 'Phase', 'Reference_peak_times_sec', 'Reference_source_or_reviewer', 'Review_notes']

FIRST 5 ROWS:


,Subject,Phase,Reference_peak_times_sec,Reference_source_or_reviewer,Review_notes
0,1,Recovery,NaN,NaN,NaN
1,1,Rest,NaN,NaN,NaN
2,1,Stress,NaN,NaN,NaN
3,2,Recovery,NaN,NaN,NaN
4,2,Rest,NaN,NaN,NaN



FILE: point11_reference_validation_and_clinical_review_sheet.csv
SHAPE: (15, 20)

COLUMNS:
['Subject', 'Phase', 'Method', 'Peak_sample', 'Peak_time_sec', 'Cleaned_amplitude', 'Matched_within_100ms', 'Nearest_NeuroKit2_time_sec', 'Distance_to_NeuroKit2_ms', 'Reviewer_label', 'Reviewer_notes', 'Suggested_review_status', 'Reference_R_peak_time_sec', 'Reference_annotator', 'Reference_source', 'Reference_review_status', 'Reviewer_decision', 'Clinical_adjudication', 'Adjudicator', 'Adjudication_notes']

FIRST 5 ROWS:


,Subject,Phase,Method,Peak_sample,Peak_time_sec,Cleaned_amplitude,Matched_within_100ms,Nearest_NeuroKit2_time_sec,Distance_to_NeuroKit2_ms,Reviewer_label,Reviewer_notes,Suggested_review_status,Reference_R_peak_time_sec,Reference_annotator,Reference_source,Reference_review_status,Reviewer_decision,Clinical_adjudication,Adjudicator,Adjudication_notes
0,76,Stress,Simple_find_peaks,64,0.64,37.150080,0,0.34,300.0,NaN,NaN,AWAITING_HUMAN_REVIEW,NaN,NaN,NaN,PENDING,PENDING,NOT_ASSESSED,NaN,NaN
1,76,Stress,Simple_find_peaks,157,1.57,30.097102,0,1.25,320.0,NaN,NaN,AWAITING_HUMAN_REVIEW,NaN,NaN,NaN,PENDING,PENDING,NOT_ASSESSED,NaN,NaN
2,76,Stress,Simple_find_peaks,252,2.52,43.223926,0,2.20,320.0,NaN,NaN,AWAITING_HUMAN_REVIEW,NaN,NaN,NaN,PENDING,PENDING,NOT_ASSESSED,NaN,NaN
3,76,Stress,Simple_find_peaks,349,3.49,26.824967,0,3.15,340.0,NaN,NaN,AWAITING_HUMAN_REVIEW,NaN,NaN,NaN,PENDING,PENDING,NOT_ASSESSED,NaN,NaN
4,76,Stress,Simple_find_peaks,434,4.34,36.765350,0,4.03,310.0,NaN,NaN,AWAITING_HUMAN_REVIEW,NaN,NaN,NaN,PENDING,PENDING,NOT_ASSESSED,NaN,NaN


In [8]:
import os
import pandas as pd

project_path = r"C:\Users\HP\MSIMI Detection"

keywords = [
    "label", "diagnosis", "msimi",
    "ischemia", "clinical", "class",
    "ground_truth", "outcome"
]

for root, dirs, files in os.walk(project_path):

    dirs[:] = [
        d for d in dirs
        if d not in ["__pycache__", ".ipynb_checkpoints"]
    ]

    for file in files:

        if file.lower().endswith(".csv"):

            path = os.path.join(root, file)

            try:
                cols = pd.read_csv(path, nrows=0).columns.tolist()

                matched = [
                    col for col in cols
                    if any(k in col.lower() for k in keywords)
                ]

                if matched:
                    print("\nFILE:", path)
                    print("MATCHING COLUMNS:", matched)

            except Exception:
                pass


FILE: C:\Users\HP\MSIMI Detection\results\point11_detector_candidates_manual_review.csv
MATCHING COLUMNS: ['Reviewer_label']

FILE: C:\Users\HP\MSIMI Detection\results\point11_reference_validation_and_clinical_review_sheet.csv
MATCHING COLUMNS: ['Reviewer_label', 'Clinical_adjudication']

FILE: C:\Users\HP\MSIMI Detection\results\point11_RR_flags_manual_adjudication.csv
MATCHING COLUMNS: ['Reviewer_beat_label']

FILE: C:\Users\HP\MSIMI Detection\results\point11_subject076_stress_extra_peak_review.csv
MATCHING COLUMNS: ['Reviewer_label']


In [9]:
import pandas as pd

files = {
    "Detector Review": r"C:\Users\HP\MSIMI Detection\results\point11_detector_candidates_manual_review.csv",

    "Clinical Review": r"C:\Users\HP\MSIMI Detection\results\point11_reference_validation_and_clinical_review_sheet.csv",

    "RR Review": r"C:\Users\HP\MSIMI Detection\results\point11_RR_flags_manual_adjudication.csv",

    "Extra Peak Review": r"C:\Users\HP\MSIMI Detection\results\point11_subject076_stress_extra_peak_review.csv"
}

for name, path in files.items():

    df_temp = pd.read_csv(path)

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    for col in df_temp.columns:
        if any(word in col.lower() for word in
               ["reviewer_label", "clinical_adjudication",
                "reviewer_beat_label"]):

            print("\nColumn:", col)
            print("Values:")
            print(df_temp[col].value_counts(dropna=False).head(20))

    display(df_temp.head(5))


Detector Review

Column: Reviewer_label
Values:
Reviewer_label
NaN    15
Name: count, dtype: int64


,Subject,Phase,Method,Peak_sample,Peak_time_sec,Cleaned_amplitude,Matched_within_100ms,Nearest_NeuroKit2_time_sec,Distance_to_NeuroKit2_ms,Reviewer_label,Reviewer_notes,Suggested_review_status
0,76,Stress,Simple_find_peaks,64,0.64,37.150080,0,0.34,300.0,NaN,NaN,AWAITING_HUMAN_REVIEW
1,76,Stress,Simple_find_peaks,157,1.57,30.097102,0,1.25,320.0,NaN,NaN,AWAITING_HUMAN_REVIEW
2,76,Stress,Simple_find_peaks,252,2.52,43.223926,0,2.20,320.0,NaN,NaN,AWAITING_HUMAN_REVIEW
3,76,Stress,Simple_find_peaks,349,3.49,26.824967,0,3.15,340.0,NaN,NaN,AWAITING_HUMAN_REVIEW
4,76,Stress,Simple_find_peaks,434,4.34,36.765350,0,4.03,310.0,NaN,NaN,AWAITING_HUMAN_REVIEW



Clinical Review

Column: Reviewer_label
Values:
Reviewer_label
NaN    15
Name: count, dtype: int64

Column: Clinical_adjudication
Values:
Clinical_adjudication
NOT_ASSESSED    15
Name: count, dtype: int64


,Subject,Phase,Method,Peak_sample,Peak_time_sec,Cleaned_amplitude,Matched_within_100ms,Nearest_NeuroKit2_time_sec,Distance_to_NeuroKit2_ms,Reviewer_label,Reviewer_notes,Suggested_review_status,Reference_R_peak_time_sec,Reference_annotator,Reference_source,Reference_review_status,Reviewer_decision,Clinical_adjudication,Adjudicator,Adjudication_notes
0,76,Stress,Simple_find_peaks,64,0.64,37.150080,0,0.34,300.0,NaN,NaN,AWAITING_HUMAN_REVIEW,NaN,NaN,NaN,PENDING,PENDING,NOT_ASSESSED,NaN,NaN
1,76,Stress,Simple_find_peaks,157,1.57,30.097102,0,1.25,320.0,NaN,NaN,AWAITING_HUMAN_REVIEW,NaN,NaN,NaN,PENDING,PENDING,NOT_ASSESSED,NaN,NaN
2,76,Stress,Simple_find_peaks,252,2.52,43.223926,0,2.20,320.0,NaN,NaN,AWAITING_HUMAN_REVIEW,NaN,NaN,NaN,PENDING,PENDING,NOT_ASSESSED,NaN,NaN
3,76,Stress,Simple_find_peaks,349,3.49,26.824967,0,3.15,340.0,NaN,NaN,AWAITING_HUMAN_REVIEW,NaN,NaN,NaN,PENDING,PENDING,NOT_ASSESSED,NaN,NaN
4,76,Stress,Simple_find_peaks,434,4.34,36.765350,0,4.03,310.0,NaN,NaN,AWAITING_HUMAN_REVIEW,NaN,NaN,NaN,PENDING,PENDING,NOT_ASSESSED,NaN,NaN



RR Review

Column: Reviewer_beat_label
Values:
Reviewer_beat_label
NaN    780
Name: count, dtype: int64


,Subject,Phase,Method,Peak_sample,Peak_time_sec,Cleaned_amplitude,Matched_within_100ms,Next_peak_time_sec,RR_interval_sec,Screening_flag,Reviewer_beat_label,Reviewer_notes,Review_status
0,3,Rest,NeuroKit2,6211,62.11,222.752747,1,64.42,2.31,OUTSIDE_0.30_2.00s,NaN,NaN,AWAITING_QUALIFIED_REVIEW
1,4,Rest,NeuroKit2,30330,303.30,174.922916,1,305.71,2.41,OUTSIDE_0.30_2.00s,NaN,NaN,AWAITING_QUALIFIED_REVIEW
2,4,Rest,NeuroKit2,31223,312.23,151.564194,1,315.16,2.93,OUTSIDE_0.30_2.00s,NaN,NaN,AWAITING_QUALIFIED_REVIEW
3,4,Rest,NeuroKit2,34711,347.11,600.518431,0,349.57,2.46,OUTSIDE_0.30_2.00s,NaN,NaN,AWAITING_QUALIFIED_REVIEW
4,5,Stress,NeuroKit2,16686,166.86,1007.284384,1,169.03,2.17,OUTSIDE_0.30_2.00s,NaN,NaN,AWAITING_QUALIFIED_REVIEW



Extra Peak Review

Column: Reviewer_label
Values:
Reviewer_label
NaN    15
Name: count, dtype: int64


,Subject,Phase,Method,Peak_sample,Peak_time_sec,Cleaned_amplitude,Matched_within_100ms,Nearest_NeuroKit2_time_sec,Distance_to_NeuroKit2_ms,Reviewer_label,Reviewer_notes
0,76,Stress,Simple_find_peaks,64,0.64,37.150080,0,0.34,300.0,NaN,NaN
1,76,Stress,Simple_find_peaks,157,1.57,30.097102,0,1.25,320.0,NaN,NaN
2,76,Stress,Simple_find_peaks,252,2.52,43.223926,0,2.20,320.0,NaN,NaN
3,76,Stress,Simple_find_peaks,349,3.49,26.824967,0,3.15,340.0,NaN,NaN
4,76,Stress,Simple_find_peaks,434,4.34,36.765350,0,4.03,310.0,NaN,NaN


In [10]:
import os

dataset_path = r"C:\Users\HP\Downloads\MSIMI_DB"

for root, dirs, files in os.walk(dataset_path):

    for file in files:
        print(os.path.join(root, file))

C:\Users\HP\Downloads\MSIMI_DB\diagnostic-criteria.json
C:\Users\HP\Downloads\MSIMI_DB\ecg_channels_description.json
C:\Users\HP\Downloads\MSIMI_DB\sub_001_baseline-data_blood-tests.json
C:\Users\HP\Downloads\MSIMI_DB\sub_001_baseline-data_chief-complaint.json
C:\Users\HP\Downloads\MSIMI_DB\sub_001_baseline-data_rest-ecg-100hz.csv
C:\Users\HP\Downloads\MSIMI_DB\sub_001_baseline-data_rest-ecg-500hz.csv
C:\Users\HP\Downloads\MSIMI_DB\sub_001_disease.json
C:\Users\HP\Downloads\MSIMI_DB\sub_001_experimental-data_petct_mbf.json
C:\Users\HP\Downloads\MSIMI_DB\sub_001_experimental-data_petct_other-indicators.json
C:\Users\HP\Downloads\MSIMI_DB\sub_001_experimental-data_petct_rest_indicators.json
C:\Users\HP\Downloads\MSIMI_DB\sub_001_experimental-data_petct_sds.json
C:\Users\HP\Downloads\MSIMI_DB\sub_001_experimental-data_recover-ecg-100hz.csv
C:\Users\HP\Downloads\MSIMI_DB\sub_001_experimental-data_recover-ecg-500hz.csv
C:\Users\HP\Downloads\MSIMI_DB\sub_001_experimental-data_stress-ecg-100h

In [12]:
import json
import os
import glob
from pprint import pprint

dataset_path = r"C:\Users\HP\Downloads\MSIMI_DB"

disease_files = glob.glob(
    os.path.join(dataset_path, "sub_*_disease.json")
)

print("Total disease files:", len(disease_files))

for path in disease_files[:10]:

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print("\nFILE:", os.path.basename(path))
    pprint(data)

Total disease files: 119

FILE: sub_001_disease.json
{'Hospital Anxiety and Depression Scale-Anxiety ': 1,
 'Hospital Anxiety and Depression Scale-Depression': 1,
 'MSIMI-based-ecg': 1,
 'Type D personality(NA≥10&SI≥10)': 1,
 'angina have no obstructive coronary artery disease': 1,
 'endothelial dysfunction': 'no description',
 'mental stressinduced myocardial ischaemia': 1,
 'microcirculatory dysfunction': 0,
 'pharmacological-induced myocardial ischaemia': 1,
 'posttraumatic stress disorder': 1}

FILE: sub_002_disease.json
{'Hospital Anxiety and Depression Scale-Anxiety ': 1,
 'Hospital Anxiety and Depression Scale-Depression': 0,
 'MSIMI-based-ecg': 0,
 'Type D personality(NA≥10&SI≥10)': 1,
 'angina have no obstructive coronary artery disease': 1,
 'endothelial dysfunction': 1,
 'mental stressinduced myocardial ischaemia': 1,
 'microcirculatory dysfunction': 0,
 'pharmacological-induced myocardial ischaemia': 0,
 'posttraumatic stress disorder': 'no description'}

FILE: sub_003_dise

In [13]:
import os
import json
import glob
import pandas as pd

dataset_path = r"C:\Users\HP\Downloads\MSIMI_DB"

disease_files = glob.glob(
    os.path.join(dataset_path, "sub_*_disease.json")
)

records = []

for path in disease_files:

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    data["subject_id"] = os.path.basename(path).replace(
        "_disease.json", ""
    )

    records.append(data)

disease_df = pd.DataFrame(records)

print("Dataset shape:", disease_df.shape)

target_columns = [
    "MSIMI-based-ecg",
    "mental stressinduced myocardial ischaemia"
]

for col in target_columns:

    print("\n==========", col, "==========")

    print(disease_df[col].value_counts(dropna=False))

    print("\nPercentage:")
    print(
        disease_df[col].value_counts(
            normalize=True, dropna=False
        ).mul(100).round(2)
    )

Dataset shape: (119, 11)

========== MSIMI-based-ecg ==========
MSIMI-based-ecg
0                 108
1                   9
no description      2
Name: count, dtype: int64

Percentage:
MSIMI-based-ecg
0                 90.76
1                  7.56
no description     1.68
Name: proportion, dtype: float64

========== mental stressinduced myocardial ischaemia ==========
mental stressinduced myocardial ischaemia
0    84
1    35
Name: count, dtype: int64

Percentage:
mental stressinduced myocardial ischaemia
0    70.59
1    29.41
Name: proportion, dtype: float64


In [14]:
import json
import pandas as pd
import os

dataset_path = r"C:\Users\HP\Downloads\MSIMI_DB"

# 1. Diagnostic criteria
criteria_path = os.path.join(
    dataset_path, "diagnostic-criteria.json"
)

with open(criteria_path, "r", encoding="utf-8") as f:
    criteria = json.load(f)

print("========== DIAGNOSTIC CRITERIA ==========")
print(json.dumps(criteria, indent=4))

# 2. Cross-tabulation
print("\n========== LABEL CROSS-TABULATION ==========")

print(pd.crosstab(
    disease_df["MSIMI-based-ecg"],
    disease_df["mental stressinduced myocardial ischaemia"],
    margins=True
))

# 3. Match disease labels with ECG features
label_df = disease_df[
    ["subject_id",
     "MSIMI-based-ecg",
     "mental stressinduced myocardial ischaemia"]
].copy()

df_merged = df_500.merge(
    label_df,
    on="subject_id",
    how="left",
    validate="one_to_one"
)

print("\n========== ECG-LABEL MATCHING ==========")

print("ECG records:", len(df_merged))
print("Matched patients:", df_merged["MSIMI-based-ecg"].notna().sum())

print("\nLabel distribution in ECG records:")
print(
    df_merged[
        "mental stressinduced myocardial ischaemia"
    ].value_counts(dropna=False)
)

print("\nHoldout-wise label distribution:")
print(pd.crosstab(
    df_merged["is_unlabelled_holdout"],
    df_merged["mental stressinduced myocardial ischaemia"],
    dropna=False
))

========== DIAGNOSTIC CRITERIA ==========
{
    "diagnostic-criteria_based-ecg": "The ECG diagnosis of MSIMI was determined by clinical physicians based on the presence of ST depression greater than 0.1 mV.",
    "diagnostic-criteria_based-petct": "MSIMI is defined as a summed difference score (SDS) \u2265 3 over the 17-segment model, which is calculated based on the difference between the summed stress score and the summed rest score via PET/CT."
}

========== LABEL CROSS-TABULATION ==========
mental stressinduced myocardial ischaemia   0   1  All
MSIMI-based-ecg                                       
0                                          82  26  108
1                                           1   8    9
no description                              1   1    2
All                                        84  35  119

========== ECG-LABEL MATCHING ==========
ECG records: 80
Matched patients: 80

Label distribution in ECG records:
mental stressinduced myocardial ischaemia
0    45
1    

In [15]:
# Convert "no description" into missing value
df_merged["MSIMI-based-ecg"] = (
    df_merged["MSIMI-based-ecg"]
    .replace("no description", np.nan)
)

# Overall ECG label distribution
print("========== ECG-BASED LABELS ==========")

print(
    df_merged["MSIMI-based-ecg"]
    .value_counts(dropna=False)
)

# Development and holdout distribution
print("\n========== HOLDOUT-WISE LABELS ==========")

print(
    pd.crosstab(
        df_merged["is_unlabelled_holdout"],
        df_merged["MSIMI-based-ecg"],
        dropna=False
    )
)

# Development dataset only
df_dev = df_merged[
    df_merged["is_unlabelled_holdout"] == False
].copy()

print("\n========== DEVELOPMENT DATA ==========")

print("Total development patients:", len(df_dev))

print(
    df_dev["MSIMI-based-ecg"]
    .value_counts(dropna=False)
)

========== ECG-BASED LABELS ==========
MSIMI-based-ecg
0.0    69
1.0     9
NaN     2
Name: count, dtype: int64

========== HOLDOUT-WISE LABELS ==========
MSIMI-based-ecg        0.0  1.0  NaN
is_unlabelled_holdout               
False                   69    9    0
True                     0    0    2

========== DEVELOPMENT DATA ==========
Total development patients: 78
MSIMI-based-ecg
0.0    69
1.0     9
Name: count, dtype: int64


C:\Users\HP\AppData\Local\Temp\ipykernel_25844\2150394812.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace("no description", np.nan)


In [17]:
import os

search_paths = [
    r"C:\Users\HP\MSIMI Detection",
    r"C:\Users\HP\Desktop\MSIMI_Project"
]

for folder in search_paths:

    print("\nSEARCHING:", folder)

    if not os.path.exists(folder):
        print("Folder not found")
        continue

    for root, dirs, files in os.walk(folder):

        for file in files:

            if file.endswith(".ipynb"):
                print(os.path.join(root, file))


SEARCHING: C:\Users\HP\MSIMI Detection
C:\Users\HP\MSIMI Detection\01_dataset_inspection.ipynb
C:\Users\HP\MSIMI Detection\02_preprocessing_qc.ipynb.ipynb
C:\Users\HP\MSIMI Detection\13_beat_window_segmentation.ipynb
C:\Users\HP\MSIMI Detection\.ipynb_checkpoints\01_dataset_inspection-checkpoint.ipynb
C:\Users\HP\MSIMI Detection\.ipynb_checkpoints\02_preprocessing_qc.ipynb-checkpoint.ipynb
C:\Users\HP\MSIMI Detection\.ipynb_checkpoints\13_beat_window_segmentation-checkpoint.ipynb
C:\Users\HP\MSIMI Detection\results\point11_comparison_plots.ipynb
C:\Users\HP\MSIMI Detection\results\.ipynb_checkpoints\point11_comparison_plots-checkpoint.ipynb
C:\Users\HP\MSIMI Detection\results\COMPARISON_POINT_11\Point_11_Comparison\point11_comparison_plots.ipynb
C:\Users\HP\MSIMI Detection\results\COMPARISON_POINT_11\Point_11_Comparison\.ipynb_checkpoints\point11_comparison_plots-checkpoint.ipynb
C:\Users\HP\MSIMI Detection\results\THREE_COMPARISON_FOLDERS\COMPARISON_POINT_11\Point_11_Comparison\point

In [18]:
import os

search_paths = [
    r"C:\Users\HP\MSIMI Detection",
    r"C:\Users\HP\Desktop\MSIMI_Project"
]

keywords = ["feature", "extract", "st", "patient_phase"]

for folder in search_paths:

    for root, dirs, files in os.walk(folder):

        dirs[:] = [
            d for d in dirs
            if d not in [".ipynb_checkpoints", "__pycache__"]
        ]

        for file in files:

            if file.endswith(".ipynb"):

                if any(k in file.lower() for k in keywords):
                    print(os.path.join(root, file))

C:\Users\HP\Desktop\MSIMI_Project\COMPARISON\05_REST_STRESS_RECOVERY\05_rest_stress_recovery_comparison.ipynb
C:\Users\HP\Desktop\MSIMI_Project\COMPARISON\07_FEATURE_COMPARISON\07_feature_comparison.ipynb
C:\Users\HP\Desktop\MSIMI_Project\notebooks\08_ecg_feature_extraction.ipynb
C:\Users\HP\Desktop\MSIMI_Project\notebooks\12_frozen_stage_visualization_gallery.ipynb
C:\Users\HP\Desktop\MSIMI_Project\notebooks\new_model_feature_selection.ipynb


In [19]:
import json

notebook_path = r"C:\Users\HP\Desktop\MSIMI_Project\notebooks\08_ecg_feature_extraction.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

keywords = [
    "st60",
    "st80",
    "adc",
    "millivolt",
    "mv",
    "st_slope"
]

for i, cell in enumerate(nb["cells"]):

    source = "".join(cell.get("source", []))

    if any(word in source.lower() for word in keywords):

        print(f"\n{'='*60}")
        print("CELL:", i)
        print("="*60)
        print(source[:4000])


CELL: 0
# 08 - ECG Feature Extraction & Spatial Repolarization Profiling
**MSIMI ECG Research Project**

### Clinical Objective & Research Context
Mental Stress-Induced Myocardial Ischemia (**MSIMI**) is characterized by transient, subtle myocardial perfusion deficits
manifesting as electrophysiological repolarization abnormalities—principally **ST-segment depression (> 0.1 mV)** during psychological challenge.
Unlike exercise-induced ischemia, mental stress triggers intense sympathetic activation accompanied by microvascular vasoconstriction,
often producing multi-lead repolarization shifts that demand sensitive spatial and temporal morphology profiling.

This notebook constructs the foundational tabular feature store across the **80-patient ANOCA cohort (`sub_001` through `sub_080`)**,
extracting multi-lead fiducials, spatial repolarization gradients, depolarization descriptors, autonomic dynamics, and phase differentials.

### Scientific Principles & Governance Safeguards
1. **Zero

In [20]:
import json

notebook_path = r"C:\Users\HP\Desktop\MSIMI_Project\notebooks\08_ecg_feature_extraction.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

keywords = [
    "st60_adc",
    "st80_adc",
    "st_slope",
    "min_st60",
    "def extract",
    "frobenius",
    "t_amp"
]

for i, cell in enumerate(nb["cells"]):

    if cell["cell_type"] != "code":
        continue

    source = "".join(cell.get("source", []))

    if any(k in source.lower() for k in keywords):

        print("\n" + "=" * 70)
        print("CELL:", i)
        print("=" * 70)
        print(source[:7000])


CELL: 6
STANDARD_12_LEADS = [
    "I", "II", "III", "aVR", "aVL", "aVF",
    "V1", "V2", "V3", "V4", "V5", "V6"
]

# 1. Explicit Lead Regional Groups
REGIONAL_GROUPS = {
    "inferior": ["II", "III", "aVF"],
    "anterior": ["V1", "V2", "V3", "V4"],
    "lateral": ["I", "aVL", "V5", "V6"],
}

# 2. Explicit Anatomical Adjacency Graph
LEAD_ADJACENCY = {
    # Precordial chain across chest wall
    "V1": ["V2"],
    "V2": ["V1", "V3"],
    "V3": ["V2", "V4"],
    "V4": ["V3", "V5"],
    "V5": ["V4", "V6"],
    "V6": ["V5"],
    # Inferior leads
    "II": ["aVF"],
    "aVF": ["II", "III"],
    "III": ["aVF"],
    # High lateral limb leads
    "I": ["aVL"],
    "aVL": ["I"],
}

def compute_contiguous_lead_count(depressed_leads: List[str], region_leads: List[str]) -> int:
    """
    Calculates the maximum number of anatomically contiguous leads with depression within a region.
    Uses connected component analysis on LEAD_ADJACENCY graph.
    """
    depressed_set = set(depressed_leads).in